# Moral judgement in DINOv2: reproduce everything

Runs the whole project in Google Colab. Features are cached in Google Drive, so each step runs once and a dropped runtime only loses the step that was running.

Before you start:
1. Accept the SMID terms at https://huggingface.co/datasets/AIML-TUDA/smid
2. Add a Hugging Face read token as a Colab secret named `HF_TOKEN` (key icon in the left bar)
3. Step 1 needs a GPU runtime (Runtime > Change runtime type). Steps 2 to 4 run on CPU.

In [ ]:
# Setup: mount Drive, get the code
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/valence_probe'          # cached features (Step 1 writes, Steps 2-4 read)
RESULTS = f'{CACHE}/results'                             # experiment outputs, kept on Drive
os.makedirs(RESULTS, exist_ok=True)
if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/eliovi/moral_judgement_in_dinov2 /content/repo
%cd /content/repo
!git pull -q

## Step 1: extract features (GPU, about 45 minutes, once)

Downloads SMID and COCO val2017, writes DINOv2 features (4 sizes, every layer), control features (image statistics, a supervised ViT) and CLIP embeddings to the cache, then deletes the images. Parts that already exist in the cache are skipped.

In [ ]:
from huggingface_hub import snapshot_download
if not os.path.exists(f'{CACHE}/features/G_smid.npy') or not os.path.exists(f'{CACHE}/features_extra/clip_img.npy'):
    snapshot_download('AIML-TUDA/smid', repo_type='dataset', local_dir='data/smid', token=userdata.get('HF_TOKEN'), max_workers=16)
    !python src/extract.py smid --smid-dir data/smid --cache {CACHE}
    !python src/extract.py controls --smid-dir data/smid --cache {CACHE}
if not os.path.exists(f'{CACHE}/coco_features/coco_clip.npy'):
    !python src/extract.py coco --coco-dir data/coco --cache {CACHE}
!rm -rf data && echo "downloaded images deleted"

## Step 2: run the experiments (CPU, about 2 to 3 hours in total)

One script per test in the technical report. Each writes its results to Drive; finished experiments are skipped, so rerun this cell if the runtime drops.

| Script | Test |
|---|---|
| `e1_probe_layers.py` | 1. Probe every layer of every DINOv2 size |
| `e2_shortcuts.py` | 2. Controls for image statistics and objects |
| `e3_beyond_valence.py` | 3. Morality beyond pleasantness: significance, noise ceiling |
| `e4_words.py` | 4. The direction in words, through a COCO-fit translation into CLIP |
| `e5_depth_scale.py` | 5. Depth and model size |
| `e6_shared.py` | 6. Shared direction between DINOv2 and CLIP |

In [ ]:
from huggingface_hub import hf_hub_download
META = hf_hub_download('AIML-TUDA/smid', 'metadata.csv', repo_type='dataset', token=userdata.get('HF_TOKEN'))
runs = [('e1_probe_layers', 'e1_probe_layers.csv', ''),
        ('e2_shortcuts', 'e2_shortcuts.json', ''),
        ('e3_beyond_valence', 'e3_beyond_valence.json', f'--metadata {META}'),
        ('e4_words', 'e4_words.json', ''),
        ('e5_depth_scale', 'e5_depth_scale.json', ''),
        ('e6_shared', 'e6_shared.json', '')]
for script, output, extra in runs:
    if os.path.exists(f'{RESULTS}/{output}'):
        print('done already:', script); continue
    !python src/{script}.py --cache {CACHE} --out {RESULTS} {extra}

## Step 3 (optional): candidate example images

Writes thumbnails of candidate images for the photo figures to `results/examples/candidates.json` on Drive. The picks used in the figures (`results/examples/picks.csv`) were chosen by hand from these: no graphic images and no identifiable public figures.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('AIML-TUDA/smid', repo_type='dataset', local_dir='data/smid', token=userdata.get('HF_TOKEN'), max_workers=16)
!python src/make_examples.py --cache {CACHE} --smid-dir data/smid --metadata {META} --out {RESULTS}
!rm -rf data/smid && echo "SMID images deleted"

## Step 4: figures

Copies your results next to the code and redraws the six figures in `figures/`.

In [ ]:
!cp {RESULTS}/e*.json {RESULTS}/e*.csv results/
!python src/figures.py
from IPython.display import Image, display
for f in sorted(os.listdir('figures')):
    display(Image(f'figures/{f}', width=900))